# NLL Stability Analysis (28 Parameter Sets)

How many datapoints and MC samples do we need for a stable likelihood estimation?

This notebook:
1. Uses the 28 uncorrelated parameter combinations from 04_parameter_recovery
2. Simulates localization responses for each parameter set
3. Evaluates NLL stability across a 2D grid:
   - **Repetitions per direction**: How many response trials per target direction
   - **MC samples**: Monte Carlo samples for the likelihood integral
4. Aggregates stability metrics across all parameter sets

All simulations and NLL evaluations are cached in `results/nll_stability_simulations.pkl`
(computed with a development version of `bayesian_listener` preceding 0.2.0 and the original
SONICOM KEMAR GRAS large-ears file); with the cache present the notebook only computes the
statistics reported in Sec. 3.3. Fig. S2 is drawn by `figures/fig_nll_stability.py`.

In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pyfar as pf
from pathlib import Path
from joblib import Parallel, delayed

from bayesian_listener import BayesianListener

import paths


def sigma2kappa(sigma):
    """Convert angular standard deviation (degrees) to von Mises-Fisher concentration."""
    return 1.0 / (2.0 * np.sin(np.deg2rad(sigma) / 2.0)**2)

## Configuration

In [2]:
# ---------- SOFA file path ----------
# KEMAR HRTF SOFA file (SONICOM dataset), shipped with the repository
sofa_path = str(paths.KEMAR_SOFA)
CACHE_DIR = paths.CACHE_DIR

# ---------- 28 Parameter combinations (from 04_parameter_recovery) ----------
# Uncorrelated, coherent sampling: sigma_spectral (2-25), sigma_prior (5-90), sigma_motor (5-16)
PARAM_COMBINATIONS = [
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.0, 'sigma_prior': 22.4, 'sigma_motor': 7.9},  # rmsP=14.2 rmsL=7.8 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.2, 'sigma_prior': 6.9, 'sigma_motor': 14.0},  # rmsP=36.1 rmsL=14.1 qe=6.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.3, 'sigma_prior': 42.5, 'sigma_motor': 5.0},  # rmsP=8.9 rmsL=5.5 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.5, 'sigma_prior': 58.7, 'sigma_motor': 15.2},  # rmsP=18.8 rmsL=15.3 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.7, 'sigma_prior': 18.1, 'sigma_motor': 13.1},  # rmsP=23.0 rmsL=13.4 qe=1.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.9, 'sigma_prior': 13.1, 'sigma_motor': 16.0},  # rmsP=33.4 rmsL=17.2 qe=2.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.1, 'sigma_prior': 20.1, 'sigma_motor': 5.4},  # rmsP=24.1 rmsL=6.7 qe=1.3
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.4, 'sigma_prior': 65.3, 'sigma_motor': 9.5},  # rmsP=20.1 rmsL=10.5 qe=4.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.6, 'sigma_prior': 30.9, 'sigma_motor': 7.0},  # rmsP=26.1 rmsL=8.1 qe=3.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.9, 'sigma_prior': 34.3, 'sigma_motor': 5.8},  # rmsP=23.5 rmsL=6.6 qe=6.4
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 4.2, 'sigma_prior': 38.2, 'sigma_motor': 14.4},  # rmsP=26.7 rmsL=14.4 qe=2.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 4.5, 'sigma_prior': 27.7, 'sigma_motor': 11.9},  # rmsP=27.9 rmsL=12.8 qe=8.8
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 4.9, 'sigma_prior': 14.6, 'sigma_motor': 9.1},  # rmsP=34.2 rmsL=9.4 qe=9.9
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 5.3, 'sigma_prior': 8.5, 'sigma_motor': 13.6},  # rmsP=41.5 rmsL=14.1 qe=13.3
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 5.7, 'sigma_prior': 52.7, 'sigma_motor': 11.1},  # rmsP=28.0 rmsL=12.7 qe=8.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 6.1, 'sigma_prior': 47.3, 'sigma_motor': 12.7},  # rmsP=33.1 rmsL=13.7 qe=13.3
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 6.6, 'sigma_prior': 6.2, 'sigma_motor': 6.6},  # rmsP=40.1 rmsL=10.3 qe=18.6
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 7.1, 'sigma_prior': 7.7, 'sigma_motor': 6.2},  # rmsP=38.6 rmsL=10.5 qe=28.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 7.7, 'sigma_prior': 24.9, 'sigma_motor': 11.5},  # rmsP=31.3 rmsL=11.8 qe=13.8
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 8.3, 'sigma_prior': 5.0, 'sigma_motor': 15.6},  # rmsP=34.4 rmsL=15.6 qe=16.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 8.9, 'sigma_prior': 5.6, 'sigma_motor': 12.3},  # rmsP=34.7 rmsL=15.1 qe=26.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 9.6, 'sigma_prior': 16.2, 'sigma_motor': 9.9},  # rmsP=35.0 rmsL=12.2 qe=18.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 10.3, 'sigma_prior': 10.6, 'sigma_motor': 8.7},  # rmsP=34.7 rmsL=11.4 qe=23.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 11.1, 'sigma_prior': 80.9, 'sigma_motor': 14.8},  # rmsP=37.4 rmsL=15.8 qe=24.1
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 12.0, 'sigma_prior': 90.0, 'sigma_motor': 10.7},  # rmsP=35.4 rmsL=12.7 qe=18.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 12.9, 'sigma_prior': 11.8, 'sigma_motor': 8.3},  # rmsP=39.3 rmsL=12.8 qe=29.1
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 13.9, 'sigma_prior': 72.7, 'sigma_motor': 7.4},  # rmsP=35.6 rmsL=10.9 qe=13.9
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 15.0, 'sigma_prior': 9.5, 'sigma_motor': 10.3},  # rmsP=43.4 rmsL=12.5 qe=24.1
]
# ---------- Simulation settings ----------
MAX_REPS_PER_TARGET = 60  # maximum trials per target direction
BASE_SEED = 42            # base seed for generating simulated responses

# ---------- 2D Sweep settings ----------
REPS_PER_TARGET = [3, 6, 9]  # repetitions per direction
MC_SAMPLES_LIST = [50, 100, 150, 200, 250, 300, 400, 500]   # MC samples for likelihood

N_BOOTSTRAP = 20          # repetitions per (reps, mc) combination
BOOTSTRAP_SEED = 123

# ---------- Parallelization ----------
N_JOBS = -1  # -1 = use all available cores

print(f'{len(PARAM_COMBINATIONS)} parameter sets')
print(f'{len(REPS_PER_TARGET)} rep counts x {len(MC_SAMPLES_LIST)} MC counts x {N_BOOTSTRAP} bootstrap = '
      f'{len(REPS_PER_TARGET) * len(MC_SAMPLES_LIST) * N_BOOTSTRAP} evaluations per param set')

28 parameter sets
3 rep counts x 8 MC counts x 20 bootstrap = 480 evaluations per param set


## Target grid

33 directions covering the full sphere (azimuths 0-330, elevations -30 to 90).

In [3]:
# (azimuth_deg, elevation_deg) -- from the experimental grid
TARGET_GRID = np.array([
    [315,  30], [315, -30], [180, -30], [ 90,  30],
    [120,   0], [ 90,   0], [225,  30], [330,   0],
    [270, -30], [  0, -30], [180,  30], [135,  30],
    [270,  30], [300,   0], [ 30,   0], [240,   0],
    [ 60,   0], [225, -30], [135, -30], [  0,  90],
    [210,   0], [270,  60], [  0,   0], [  0,  30],
    [ 90,  60], [180,  60], [180,   0], [270,   0],
    [ 45,  30], [ 45, -30], [  0,  60], [150,   0],
    [ 90, -30],
])

_tg_arr = np.column_stack([
    np.deg2rad(TARGET_GRID[:, 0]),
    np.deg2rad(TARGET_GRID[:, 1]),
    np.ones(len(TARGET_GRID)),
])
target_coords = pf.Coordinates.from_spherical_elevation(_tg_arr[:, 0], _tg_arr[:, 1], _tg_arr[:, 2])

print(f'{len(TARGET_GRID)} target directions')

33 target directions


## Load model and prepare features

In [4]:
model = BayesianListener(sofa_path)
model.compute_template(interpolation='SHMAX', cache_dir=CACHE_DIR)

# Map the 33 target directions to closest positions in the model grid
target_indices = model.target.coords.find_nearest(target_coords)[0][0]
targets = model.target[target_indices]

print(f'Target feature matrix: {targets.features.shape}')
print(f'Model loaded, will iterate over {len(PARAM_COMBINATIONS)} parameter sets')

Target feature matrix: (33, 58)
Model loaded, will iterate over 28 parameter sets


## Simulate responses for all parameter sets

Generate synthetic localization data for each of the 28 parameter combinations.

In [5]:
n_targets = len(TARGET_GRID)
n_params = len(PARAM_COMBINATIONS)

save_path = paths.RESULTS_DIR / 'nll_stability_simulations.pkl'

if save_path.exists():
    print(f'Loading cached data from {save_path.relative_to(paths.ROOT)}...')
    with open(save_path, 'rb') as f:  # only unpickle files from trusted sources
        cached = pickle.load(f)
    all_responses = cached['all_responses']
    all_resp_targets_idx = cached['all_resp_targets_idx']
    all_doa_estimations = cached['all_doa_estimations']
    all_kappa_motor = cached['all_kappa_motor']
    # Load nll_results if available
    nll_results = cached.get('nll_results', None)
    n_total = len(all_responses[0])
    print(f'Loaded {n_total} responses per param set ({n_params} sets)')
    print(f'Predictions shape per set: {all_doa_estimations[0].shape}')
    if nll_results is not None:
        print(f'NLL results shape: {nll_results.shape}')
    else:
        print('NLL results not cached — run sweep cells below')
else:
    def simulate_responses_for_param(p_idx, params, sofa_path, targets, max_reps, base_seed):
        """Simulate responses for a single parameter set."""
        from bayesian_listener import BayesianListener

        model = BayesianListener(sofa_path)
        model.compute_template(interpolation='SHMAX', cache_dir=CACHE_DIR)
        model.target = targets

        seed = base_seed + p_idx * 1000
        kappa_motor = sigma2kappa(params['sigma_motor'])
        model.parameters = {k: v for k, v in params.items() if k != 'sigma_motor'} | {'kappa_motor': kappa_motor}
        posterior_idx = model.infer(repetitions=max_reps, seed=seed)
        estimations = model.estimate(posterior_idx, kappa_motor=kappa_motor, seed=seed).cartesian

        # Flatten to (n_total, 3) format
        responses_cart = estimations.reshape(-1, 3)
        resp_targets_idx = np.repeat(np.arange(len(targets)), max_reps)

        # Normalise to unit vectors
        norms = np.linalg.norm(responses_cart, axis=1, keepdims=True)
        responses_cart = responses_cart / norms

        return p_idx, responses_cart, resp_targets_idx

    print(f'Simulating responses for {n_params} parameter sets in parallel...')

    results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(simulate_responses_for_param)(
            p_idx, params, sofa_path, targets, MAX_REPS_PER_TARGET, BASE_SEED
        )
        for p_idx, params in enumerate(PARAM_COMBINATIONS)
    )

    # Collect results
    all_responses = {}
    all_resp_targets_idx = {}
    for p_idx, responses_cart, resp_targets_idx in results:
        all_responses[p_idx] = responses_cart
        all_resp_targets_idx[p_idx] = resp_targets_idx

    n_total = len(all_responses[0])
    print(f'\nSimulated {n_total} responses per param set ({n_params} sets)')
    print(f'Total: {n_total * n_params} responses')

    # Placeholders — will be filled by cell 11
    all_doa_estimations = None
    all_kappa_motor = None
    nll_results = None

Loading cached data from results/nll_stability_simulations.pkl...
Loaded 1980 responses per param set (28 sets)
Predictions shape per set: (33, 500, 3)
NLL results shape: (28, 3, 8, 20)


## Pre-compute model predictions for all parameter sets

Run Monte Carlo inference at each MC sample count for each parameter set.

In [6]:
# Pre-compute predictions and kappa values for each parameter set
if all_doa_estimations is not None:
    print(f'Predictions already loaded from cache — skipping computation')
    print(f'Shape per set: {all_doa_estimations[0].shape}')
else:
    max_mc = max(MC_SAMPLES_LIST)

    def precompute_predictions_for_param(p_idx, params, sofa_path, targets, max_mc, base_seed):
        """Pre-compute model predictions for a single parameter set."""
        from bayesian_listener import BayesianListener

        model = BayesianListener(sofa_path)
        model.compute_template(interpolation='SHMAX', cache_dir=CACHE_DIR)
        model.target = targets

        # Set parameters (motor noise is disabled in estimate() below)
        model.parameters = {
            'sigma_itd': params['sigma_itd'],
            'sigma_ild': params['sigma_ild'],
            'sigma_spectral': params['sigma_spectral'],
            'sigma_prior': params['sigma_prior'],
            'kappa_motor': sigma2kappa(params['sigma_motor']),
        }

        seed = base_seed + p_idx * 1000 + 500
        posterior_mc = model.infer(repetitions=max_mc, seed=seed)
        doa_estimations = model.estimate(posterior_mc, kappa_motor=False).cartesian
        kappa_motor = sigma2kappa(params['sigma_motor'])

        return p_idx, doa_estimations, kappa_motor

    print(f'Pre-computing predictions for {n_params} parameter sets in parallel...')

    results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(precompute_predictions_for_param)(
            p_idx, params, sofa_path, targets, max_mc, BASE_SEED
        )
        for p_idx, params in enumerate(PARAM_COMBINATIONS)
    )

    # Collect results
    all_doa_estimations = {}
    all_kappa_motor = {}
    for p_idx, doa_estimations, kappa_motor in results:
        all_doa_estimations[p_idx] = doa_estimations
        all_kappa_motor[p_idx] = kappa_motor

    print(f'\nPre-computed predictions for {len(PARAM_COMBINATIONS)} parameter sets')
    print(f'Shape per set: {all_doa_estimations[0].shape}')

Predictions already loaded from cache — skipping computation
Shape per set: (33, 500, 3)


## Helper functions

In [7]:
# Save simulated data (skip if already loaded from cache)
if save_path.exists():
    print(f'Cache already exists at {save_path.relative_to(paths.ROOT)} — skipping save')
else:
    data_to_save = {
        'all_responses': all_responses,
        'all_resp_targets_idx': all_resp_targets_idx,
        'all_doa_estimations': all_doa_estimations,
        'all_kappa_motor': all_kappa_motor,
        'PARAM_COMBINATIONS': PARAM_COMBINATIONS,
        'TARGET_GRID': TARGET_GRID,
        'REPS_PER_TARGET': REPS_PER_TARGET,
        'MC_SAMPLES_LIST': MC_SAMPLES_LIST,
        'N_BOOTSTRAP': N_BOOTSTRAP,
        'MAX_REPS_PER_TARGET': MAX_REPS_PER_TARGET,
        'BASE_SEED': BASE_SEED,
        'BOOTSTRAP_SEED': BOOTSTRAP_SEED,
    }
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, 'wb') as f:
        pickle.dump(data_to_save, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'Saved to {save_path.relative_to(paths.ROOT)} ({save_path.stat().st_size / 1e6:.1f} MB)')

Cache already exists at results/nll_stability_simulations.pkl — skipping save


In [8]:
def fast_nll(doa_estimations, kappa, responses_cart, resp_targets_idx):
    """
    Evaluate negative log-likelihood using pre-computed model predictions.

    Uses von Mises-Fisher PDF to compare observed responses against
    Monte Carlo model predictions, averaged over MC samples.
    """
    if kappa < 1e-6:
        C = 1.0 / (4.0 * np.pi)
    else:
        C = kappa / (4.0 * np.pi * np.sinh(kappa))

    loglik_total = 0.0

    for target_idx in np.unique(resp_targets_idx):
        obs_mask = resp_targets_idx == target_idx
        obs = responses_cart[obs_mask, :]
        est = doa_estimations[target_idx, :, :]

        cos_angles = obs @ est.T
        log_pdfs = np.log(C) + kappa * cos_angles

        # Log-sum-exp trick: average over MC samples
        max_log_pdfs = np.max(log_pdfs, axis=1, keepdims=True)
        log_mean_probs = (max_log_pdfs.squeeze()
                          + np.log(np.mean(np.exp(log_pdfs - max_log_pdfs), axis=1)))

        loglik_total += np.sum(log_mean_probs)

    return -loglik_total


def subsample_responses(resp_targets_idx, reps_per_target, rng):
    """
    Subsample exactly `reps_per_target` responses for each target direction.

    Returns indices into the full response array.
    """
    unique_targets = np.unique(resp_targets_idx)
    indices = []

    for t in unique_targets:
        pool = np.where(resp_targets_idx == t)[0]
        chosen = rng.choice(pool, size=reps_per_target, replace=False)
        indices.append(chosen)

    return np.concatenate(indices)


def compute_single_nll(p_idx, i, j, b, reps, mc_samples,
                       all_doa_estimations, all_kappa_motor,
                       all_responses, all_resp_targets_idx, bootstrap_seed):
    """
    Compute NLL for a single (param_set, reps, mc_samples, bootstrap) combination.
    """
    rng = np.random.default_rng(bootstrap_seed + p_idx * 100000 + i * 10000 + j * 100 + b)

    # Get data for this parameter set
    doa_estimations_full = all_doa_estimations[p_idx]
    kappa_motor = all_kappa_motor[p_idx]
    responses_cart = all_responses[p_idx]
    resp_targets_idx = all_resp_targets_idx[p_idx]

    # Slice MC samples from pre-computed predictions
    doa_estimations = doa_estimations_full[:, :mc_samples, :]

    # Subsample responses (reps per target direction)
    sub_idx = subsample_responses(resp_targets_idx, reps, rng)

    # Compute NLL
    nll = fast_nll(doa_estimations, kappa_motor,
                   responses_cart[sub_idx], resp_targets_idx[sub_idx])

    return p_idx, i, j, b, nll

## Run 2D NLL stability sweep across all parameter sets

Iterate over all combinations of (parameter set, repetitions per target, MC samples) and compute NLL statistics.

In [9]:
n_reps = len(REPS_PER_TARGET)
n_mc = len(MC_SAMPLES_LIST)
n_params = len(PARAM_COMBINATIONS)

if nll_results is not None:
    print(f'NLL results already loaded from cache — skipping sweep')
    print(f'Results shape: {nll_results.shape}')
else:
    n_total_jobs = n_params * n_reps * n_mc * N_BOOTSTRAP
    print(f'Running {n_total_jobs} NLL evaluations with {N_JOBS} workers...')

    # Build list of all jobs
    jobs = [
        (p_idx, i, j, b, reps, mc_samples)
        for p_idx in range(n_params)
        for i, reps in enumerate(REPS_PER_TARGET)
        for j, mc_samples in enumerate(MC_SAMPLES_LIST)
        for b in range(N_BOOTSTRAP)
    ]

    # Run in parallel
    results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(compute_single_nll)(
            p_idx, i, j, b, reps, mc_samples,
            all_doa_estimations, all_kappa_motor,
            all_responses, all_resp_targets_idx,
            BOOTSTRAP_SEED
        )
        for p_idx, i, j, b, reps, mc_samples in jobs
    )

    # Collect results into array: (n_params, n_reps, n_mc, N_BOOTSTRAP)
    nll_results = np.zeros((n_params, n_reps, n_mc, N_BOOTSTRAP))
    for p_idx, i, j, b, nll in results:
        nll_results[p_idx, i, j, b] = nll

    print(f'\nResults shape: {nll_results.shape}')

    # Update cache with nll_results
    with open(save_path, 'rb') as f:
        cached = pickle.load(f)
    cached['nll_results'] = nll_results
    with open(save_path, 'wb') as f:
        pickle.dump(cached, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'Updated cache with nll_results at {save_path}')

NLL results already loaded from cache — skipping sweep
Results shape: (28, 3, 8, 20)


## Compute statistics

Per parameter set, the standard deviation of the NLL across bootstrap replicates is scaled to a
per-trial value (divided by √N, N = number of observed trials), then summarised across the 28
parameter sets. These are the values plotted in Fig. S2 and quoted in Sec. 3.3.

In [10]:
n_targets = len(TARGET_GRID)
n_obs_per_rep = np.array(REPS_PER_TARGET) * n_targets                       # (n_reps,)

# nll_results: (n_params, n_reps, n_mc, N_BOOTSTRAP)
std_per_trial = nll_results.std(axis=3) / np.sqrt(n_obs_per_rep)[None, :, None]  # (n_params, n_reps, n_mc)
std_per_trial_mean = std_per_trial.mean(axis=0)   # (n_reps, n_mc)
std_per_trial_sd = std_per_trial.std(axis=0)      # (n_reps, n_mc), across parameter sets

print('Per-trial NLL SD: mean ± SD across the 28 parameter sets')
print(f"{'MC samples':>12}" + ''.join(f'{f"{r} reps/dir":>18}' for r in REPS_PER_TARGET))
for j, mc in enumerate(MC_SAMPLES_LIST):
    print(f'{mc:>12}' + ''.join(f'{f"{std_per_trial_mean[i, j]:.2f} ± {std_per_trial_sd[i, j]:.2f}":>18}'
                                for i in range(len(REPS_PER_TARGET))))

# Values reported in Sec. 3.3 (3 repetitions per direction, 50 -> 200 MC samples)
i_rep, j50, j200 = 0, MC_SAMPLES_LIST.index(50), MC_SAMPLES_LIST.index(200)
m50, s50 = std_per_trial_mean[i_rep, j50], std_per_trial_sd[i_rep, j50]
m200, s200 = std_per_trial_mean[i_rep, j200], std_per_trial_sd[i_rep, j200]
print(f'\n{REPS_PER_TARGET[i_rep]} reps/dir: {m50:.2f} ± {s50:.2f} (50 MC) -> {m200:.2f} ± {s200:.2f} (200 MC), '
      f'reduction {100 * (1 - m200 / m50):.0f}%')

Per-trial NLL SD: mean ± SD across the 28 parameter sets
  MC samples        3 reps/dir        6 reps/dir        9 reps/dir
          50       4.31 ± 3.86       4.11 ± 3.92       3.81 ± 2.71
         100       2.93 ± 2.37       2.56 ± 1.52       2.84 ± 1.90
         150       2.47 ± 1.29       2.16 ± 1.07       2.10 ± 0.91
         200       2.01 ± 0.85       2.02 ± 0.63       1.93 ± 0.82
         250       1.90 ± 0.86       2.01 ± 0.86       1.90 ± 0.81
         300       1.84 ± 0.59       1.74 ± 0.58       1.93 ± 0.62
         400       1.70 ± 0.25       1.71 ± 0.45       1.67 ± 0.39
         500       1.68 ± 0.25       1.81 ± 0.58       1.61 ± 0.51

3 reps/dir: 4.31 ± 3.86 (50 MC) -> 2.01 ± 0.85 (200 MC), reduction 53%


## Paper figure

Fig. S2 (per-trial NLL SD vs Monte Carlo samples, error bars = SD / √28) is drawn from the same
cache by `figures/fig_nll_stability.py`.